# NOTEBOOK USED FOR TROUBLESHOOTING RIGS

Modified from notebooks\BehaviorAnalysis\General\SummaryTodayTraining.ipynb

In [ ]:
# DEFINITIONS CELL
from aind_vr_foraging_analysis.utils.parsing import data_access
from aind_vr_foraging_analysis.utils.parsing.parse import ContinuousData


# Libraries
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

verbose = False

In [ ]:
# FUNCTIONS

# LOAD AND PARSE DATA

def load_session(path: Path) -> list[pd.DataFrame, ContinuousData, dict, pd.DataFrame, pd.DataFrame]:
    """Gets all the datastreams from a directory"""
    try:
        all_epochs, stream_data, data = data_access.load_session(path, extra=True)
        odor_sites = all_epochs.loc[all_epochs['label'] == 'OdorSite']
        odor_triggers = stream_data.odor_triggers
    except Exception as e:
            print(f"Error loading a behavior session in {path.name}: {e}")
    return all_epochs, stream_data, data, odor_sites, odor_triggers

def load_streams_from_harp(data:dict, harp:str, list_of_stream_names:list[str], drop:list[str] = ["MessageType"]) -> dict[str, dict[str, str]]:
    """
    Collect a list of data streams from a specific harp device.

    Parameters
    ----------
    data : dict
        Dictionary of harp devices (e.g., data["harpX"]).
    stream_names : list[str]
        List of stream names to extract (e.g., ["WhoAmI", "HardwareVersionHigh"]).
    drop : List[str] | None (optional)
        List of columns to not collect

    Returns
    -------
    dict
        Nested dictionary: {harp_name: {stream_name: stream_values}}
    """
    stream_dict = {}
    for stream_name in list_of_stream_names:
        data[harp].streams[stream_name].load_from_file()
        stream_dict[stream_name]= data[harp].streams[stream_name].data.drop(columns=drop, errors="ignore")
    return stream_dict

def preload_some_streams(data: dict) -> dict[str, dict[str, str]]:
    """  Function used to preload the streams that will be used later    """
    
    olf_streams_lst = ["Flowmeter", "EndValveState", "OdorValveState",
                "Channel0ActualFlow", "Channel1ActualFlow", "Channel2ActualFlow", "Channel3ActualFlow","Channel4ActualFlow"]
    treadmill_streams_lst = ["Encoder", "Torque", "SensorData"]
    evnts_streams_lst= ["ActivePatch", "ActiveSite", "ArmOdor", "GlobalPatchState", "PatchStateAtReward"]

    envmt_stream = load_streams_from_harp(data, "harp_environment_sensor", ["SensorData"])  
    olf_streams = load_streams_from_harp(data, "harp_olfactometer", olf_streams_lst)        
    treadmill_streams = load_streams_from_harp(data, "harp_treadmill", treadmill_streams_lst)
    sw_events_streams = load_streams_from_harp(data, "software_events", evnts_streams_lst, ["dataType", "data_type_hint", "timestamp_source"])
    bhv_events =  load_streams_from_harp(data, "harp_behavior", ["PulseSupplyPort0"])
    
    return envmt_stream, olf_streams, treadmill_streams, sw_events_streams, bhv_events

def collect_value_across_all_harps(data: dict, stream_name_list: list[str]) -> dict:
    """
    Collect individual value from specified streams across all harp devices.

    Parameters
    ----------
    data : dict
        Dictionary of harp devices (e.g., data["harpX"]).
    stream_names : list[str]
        List of stream names to extract (e.g., ["WhoAmI", "HardwareVersionHigh"]).

    Returns
    -------
    dict
        Nested dictionary: {harp_name: {stream_name: stream_value}}
    """
    results = {}

    for harp_name, harp_obj in data.items():
        if harp_name.startswith("harp"):
            harp_results = {}
            for stream in stream_name_list:
                try:
                    stream_obj = harp_obj.streams[stream]
                    # Try load_from_file first
                    value = stream_obj.load_from_file()
                    if value is None or hasattr(value, "empty"):
                        # Fallback to .data
                        value = stream_obj.data
                    # If it's a DataFrame, extract the first value
                    if hasattr(value, "iloc"):
                        try:
                            scalar = value.iloc[0, 0]
                        except Exception:
                            scalar = value
                        harp_results[stream] = scalar
                    else:
                        harp_results[stream] = value
                except Exception as e:
                    harp_results[stream] = f"Error: {e}"
            results[harp_name] = harp_results
    return results

#-----------------------------------------------------

# METRICS CALCULATION AND DATA SLICING

def slice_stream_in_equal_slices(df: pd.DataFrame, n: int, func, decimals: int | None = None, normalize: bool = True) -> dict[str, float]:
    """Split df into n equal-duration time windows using the float index (seconds), compute a metric per window (and 'Full'), optionally normalize by minutes."""
    if n <= 0: raise ValueError("n must be > 0")
    if df.empty: raise ValueError("df must not be empty")

    def _to_scalar(x) -> int|float:
        if isinstance(x, (pd.Series, pd.DataFrame)):
            return x.mean().mean()
        return x

    start = float(df.index.min())
    end = float(df.index.max())
    if not np.isfinite(start) or not np.isfinite(end):
        raise ValueError("Index must contain finite numeric timestamps")
    total_span = end - start
    if total_span <= 0:
        raise ValueError("Index span must be positive (max > min)")
    total_minutes = total_span / 60.0

    results: dict[str, float] = {}

    # Full dataset
    full_raw = _to_scalar(func(df))
    denom = total_minutes if normalize else 1.0
    full_val = full_raw / denom
    if decimals is not None:
        full_val = round(float(full_val), decimals)
    results["Full"] = full_val
    if verbose:     
        print(f"Full dataset: {full_val} (computed as {full_raw} / minutes={total_minutes})") if normalize else print(f"Full dataset: {full_val}")

    # Equal-duration time windows (based on index range)
    edges = np.linspace(start, end, n + 1)
    for i in range(n):
        left, right = float(edges[i]), float(edges[i + 1])
        if right <= left:
            continue

        mask = (df.index >= left) & (df.index < right if i < n - 1 else df.index <= right)
        part = df.loc[mask]
        if part.empty:
            continue

        window_minutes = (right - left) / 60.0
        slice_raw = _to_scalar(func(part))
        denom = window_minutes if normalize else 1.0
        val = round(slice_raw / denom, decimals)

        start_pct = int(round(i * 100 / n))
        end_pct = int(round((i + 1) * 100 / n))

        label = f"{start_pct}-{end_pct}%"
        if verbose:     
            print(f"Slice dataset: {val} (computed as {slice_raw} / {(window_minutes)}min)") if normalize else print(f"Slice dataset: {val}")
        results[label] = val

    return results

def slice_stream_by_minutes(df: pd.DataFrame, window_minutes: float, func, decimals: int | None = None, normalize: bool = True) -> dict[str, float]:
    """Split df into fixed-duration windows of `window_minutes` (index in seconds), compute a metric per window and 'Full', optionally normalize by minutes."""
    if window_minutes <= 0:
        raise ValueError("Error while trying to create the slices of a metric. Window_minutes must be > 0")
    if df.empty:
        raise ValueError(f"Error while trying to create the slices of a metric. DataFrame must not be empty")

    def _to_scalar(x) -> float:
        if isinstance(x, (pd.Series, pd.DataFrame)):
            return x.mean().mean()
        return x

    def _fmt(m) -> str:
        return str(int(m)) if float(m).is_integer() else f"{m:.1f}".rstrip("0").rstrip(".")

    start = float(df.index.min())
    end = float(df.index.max())
    if not np.isfinite(start) or not np.isfinite(end):
        raise ValueError("Index must contain finite numeric timestamps (seconds)")
    total_span = end - start
    if total_span <= 0:
        raise ValueError("Index span must be positive (max > min)")
    total_minutes = total_span / 60.0

    results: dict[str, float] = {}

    # Full dataset
    full_raw = _to_scalar(func(df))
    denom = total_minutes if normalize else 1.0
    full_val = full_raw / denom
    if decimals is not None:
        full_val = round(float(full_val), decimals)
    results["Full"] = full_val
    if verbose:     
        print(f"Full dataset: {full_val} (computed as {full_raw} / minutes={total_minutes})") if normalize else print(f"Full dataset: {full_val}")

    # Fixed-duration windows
    start_min = 0.0
    while start_min < total_minutes:
        end_min = start_min + window_minutes
        left = start + start_min * 60.0
        right = start + end_min * 60.0

        # Include right edge only for the last window
        is_last = end_min >= total_minutes
        mask = (df.index >= left) & (df.index < right if not is_last else df.index <= right)
        part = df.loc[mask]
        if not part.empty:
            duration_minutes = (right - left) / 60.0
            if duration_minutes > 0:
                slice_raw = _to_scalar(func(part))
                denom = duration_minutes if normalize else 1.0
                val = slice_raw / denom
                if decimals is not None:
                    val = round(float(val), decimals)
                label = f"{_fmt(start_min)}-{_fmt(end_min)}"
                if verbose:     
                    print(f"Slice dataset: {val} (computed as {slice_raw} / {duration_minutes}min)") if normalize else print(f"Slice dataset: {val}")
                
                results[label] = val

        start_min = end_min

    return results

def summarize_task(data, odor_sites, envmt_stream, olf_streams, treadmill_streams) -> dict [str]:

    def merge_dicts(**dicts) -> dict:
        from collections import defaultdict
        merged = defaultdict(dict)
        for name, d in dicts.items():
            for slice_key, value in d.items():
                merged[slice_key][name] = value
        return dict(merged)

    # Odor sites
    total_harvest_attempts = odor_sites.loc[(odor_sites['is_choice']==True)]['reward_available'].count()
    rewarded = odor_sites.loc[odor_sites.is_reward==1]['is_choice'].count()
    rewarded_perc = round((rewarded/total_harvest_attempts)*100,2)
    unrewarded = odor_sites.loc[(odor_sites.is_reward==0 ) & (odor_sites['is_choice']==True)]['is_choice'].count()
    unrewarded_perc = round((unrewarded/total_harvest_attempts)*100,2)
    water_collected = odor_sites.loc[(odor_sites['is_reward']==1)]['reward_amount'].sum()
    
    # Operation control
    total_stops = data["operation_control"].streams["IsStopped"].data["IsStopped"].sum()
    stops = data["operation_control"].streams["IsStopped"].data["IsStopped"].reset_index()
    stops['index_diff'] = abs(stops['Seconds'].diff(-1).fillna(0))
    stopped_t = stops.groupby('IsStopped')['index_diff'].sum().iloc[1]

    seconds_per_stop = stops.groupby('IsStopped')['index_diff'].nth(1).agg(['max', 'min', 'mean', 'std']).to_dict()
    
    total_travelled = data["operation_control"].streams["CurrentPosition"].data["Position"].max().round(0)/100
    
    # Odor-specific water amounts
    odor_rewards = {
        odor_label: odor_sites.loc[
            (odor_sites['odor_label'] == odor_label) & (odor_sites['is_reward'] == 1),
            'reward_amount'
        ].sum()
        for odor_label in odor_sites.odor_label.unique()
    }

    #Behavior data
    bhv_dict = data["harp_behavior"].streams["PulseSupplyPort0"].data.iloc[1:, 0].agg(['max', 'min', 'mean', 'std']).to_dict()

    # Environment data
    envmt_cols = ["Pressure", "Temperature", "Humidity"]
    envmt_dict = envmt_stream["SensorData"].loc[:, envmt_cols].agg(['max', 'min', 'mean', 'std']).to_dict()
    
    # Olfactometer data
    olf_cols = [ "Channel0ActualFlow", "Channel1ActualFlow", "Channel2ActualFlow", "Channel3ActualFlow", "Channel4ActualFlow"]
    olfactometer_dict = pd.concat([olf_streams[col][[col]] for col in olf_cols], axis=1).agg(['max', 'min', 'mean', 'std']).to_dict()
    
    # Treadmill data
    tdmll_cols = ["Encoder", "Torque", "TorqueLoadCurrent", "velocity", "distance", "filtered_velocity"]
    treadmill_dict = treadmill_streams["SensorData"].loc[:, tdmll_cols].agg(['max', 'min', 'mean', 'std']).to_dict()

    #Using encoder data to calculate the session duration, as it has data across the whole session without interruptions (like the olfactometer streams)
    time_sec = float(treadmill_streams["SensorData"]["Encoder"].index.max() - treadmill_streams["SensorData"]["Encoder"].index.min())
    minutes = round(time_sec / 60 , 2)

    
    # Slices
    n = 5 #Minutes for each slice
    slice_dict = merge_dicts(
        rewarded_stops=slice_stream_by_minutes(odor_sites['is_choice'], n, sum, decimals=2, normalize=False),
        unrewarded_stops=slice_stream_by_minutes(odor_sites.loc[(odor_sites.is_reward==0 ) & (odor_sites['is_choice']==True)]['is_choice'], n, sum, decimals=2, normalize=False),
        total_harvest_attempts=slice_stream_by_minutes(odor_sites.loc[odor_sites['is_choice']==True]['reward_available'], n, lambda x: x.count(), decimals=2, normalize=False),
        total_stops=slice_stream_by_minutes(data["operation_control"].streams["IsStopped"].data["IsStopped"], n, sum, decimals=2, normalize=False),
        water_collected=slice_stream_by_minutes(odor_sites.loc[odor_sites['is_reward']==1]['reward_amount'], n, sum, decimals=2, normalize=False),
        travelled=slice_stream_by_minutes((data["operation_control"].streams["CurrentPosition"].data["Position"]/100), n, lambda x: x.max()-x.min(), decimals=2, normalize=False)
    )

    # Final dictionary
    summary = {
        "task": {
            "minutes": minutes,
            "total_sites/min": len(odor_sites)/minutes,
            "total_harvest_attempts/min": total_harvest_attempts/minutes,
            "total_stops/min": total_stops/minutes,
            "rewarded_stops": rewarded/minutes,
            "rewarded_stops_perc": rewarded_perc,
            "unrewarded_stops": unrewarded/minutes,
            "unrewarded_stops_perc": unrewarded_perc,
            "water_collected_ul": water_collected,
            "total_travelled_m": total_travelled,
            # "odor_rewards": odor_rewards,
            "stopped_t": stopped_t
        },
        "seconds_per_stop": seconds_per_stop,
        "environment": envmt_dict,
        "olfactometer": olfactometer_dict,
        "treadmill": treadmill_dict,
        "behavior": bhv_dict,
        "Slices": slice_dict
    }
    print(summary)
    return summary


#-----------------------------------------------------

#SAVING DATA
def save_json(dict:dict, path:str) -> None:
    with open(os.path.join(path,'summary.json'), 'w') as fp:
        json.dump(dict, fp, indent=4, default=int)

#-----------------------------------------------------
# PLOTTING HELPERS
def annotate_points(ax: plt.Axes, x: list, y: list, tags: list | None, fontsize: int = 15, color: str = 'black') -> None:
    """Annotate (x, y) points with text tags on a Matplotlib Axes."""
    if tags is not None:
        for xi, yi, tag in zip(x, y, tags):
            if pd.notna(yi):
                ax.annotate(str(tag), (xi, yi), textcoords="offset points", xytext=(3, 5), fontsize=fontsize, alpha=0.9, color=color)


# **One session exploration**

In [ ]:
# Generates the summary in a single session
path = Path(r"F:\Data\828425\828425_2026-02-19T210756Z")

all_epochs, stream_data, data, odor_sites, odor_triggers = load_session(path)

envmt_stream, olf_streams, treadmill_streams, sw_events_streams, bhv_events = preload_some_streams(data)
summary = summarize_task(data, odor_sites,envmt_stream, olf_streams, treadmill_streams)
summary["versions"] = collect_value_across_all_harps(data, ["HardwareVersionHigh", "HardwareVersionLow", "AssemblyVersion", "FirmwareVersionHigh", "FirmwareVersionLow"])
save_json(summary, path)

In [ ]:
#Run this cell for more info about the available datastreams and how to access them
# Also see: https://allenneuraldynamics.github.io/Aind.Behavior.VrForaging/dataset.html 
print("Contents in data available:")
for key, value in data.items():
    print(key, "-",value)

print(f"--------------------\nContents from the '{key}' DataStreamSource-> {data[key].streams} \n-----------------------" )


print("The DataStreamSource is iterable and will contain DataStreamType.XXXX streams:")
for element in data[key].streams:    
    print(data[key].streams[element])  

(f"If trying to access a stream results in this message: 'DataStreamType.XXXX' stream with None/Not loaded entries\n", 
      "You need to load it first using .load_from_file() method, or access through the property .data, that will load it too")
(f"data[key].streams[element].data or data[key].streams.element.data can be used. example:", data[key].streams[element].data)


In [ ]:
# Extra cell used to check a specific DataStream if needed
data['harp_behavior'].streams["PulseSupplyPort0"].load_from_file()
# data["software_events"].streams
data["harp_behavior"].streams["PulseSupplyPort0"].data.iloc[1:, 0]



# Multi Sessions in the same directory

RUN THE FOLLOWING CELL FOR SAVING A SUMMARY OF THE SESSION IN EVERY FOLDER OF A GIVEN PATH

In [ ]:
main_path = Path(r"C:/Data/tests")

rewrite = False # Set to False to skip analysis if summary.json already exists in session folder

dirs = [os.path.join(main_path, name) for name in os.listdir(main_path) if os.path.isdir(os.path.join(main_path, name))]
print(dirs)


for path in dirs:
    summary_path = os.path.join(path, "summary.json")
    if os.path.exists(summary_path) and not rewrite:
        print(f"{summary_path} already exists. Skipping analysis")
        continue
    
    try:
        all_epochs, stream_data, data, odor_sites, odor_triggers = load_session(path)
        envmt_stream, olf_streams, treadmill_streams, sw_events_streams, bhv_events = preload_some_streams(data)
        summary = summarize_task(data, odor_sites,envmt_stream, olf_streams, treadmill_streams)
        save_json(summary, path)
        print(f"Analysis made and stored as {summary_path}")
    except Exception as e:
        print(f"Not possible to generate {summary_path}>{e}")

USE THE FOLLOWING CELL TO LOAD ALL SUMMARIES AND PLOT EVERYTHING

In [ ]:
dir = Path(r"C:/Data/")
tests_dir = Path(os.path.join(dir, "Tests"))
save_img_path = None
# save_img_path = os.path.join(dir, "img/") # Comment to skip saving

# Load Summaries
def load_json_summaries(tests_dir:Path) -> pd.DataFrame:
    
    def flatten_json(d: dict, parent_key: str = "", sep: str = "_") -> dict:
        items = []
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k

            if isinstance(v, dict):
                items.extend(flatten_json(v, new_key, sep=sep).items())
            else:
                items.append((new_key, v))

        return dict(items)
    
    rows = []
    for session_dir in tests_dir.iterdir():
        acq_file = session_dir / "acquisition.json"
        acq_file2 = session_dir / "acquisition_vrforaging.json"
        summary_file = session_dir / "summary.json"

        if not summary_file.exists():
            continue
        
        if acq_file.exists():
            with open(acq_file) as f:
                acq = json.load(f)
        elif acq_file2.exists():
            with open(acq_file2) as f:
                acq = json.load(f)    
        else:
            continue  
        
        with open(summary_file) as f:
            summary = json.load(f)

        # --- Session time ---
        start = pd.Timestamp(acq["acquisition_start_time"])
        end = pd.Timestamp(acq["acquisition_end_time"])
        duration_min = (end - start).total_seconds() / 60
        day = start.date()
        
        # --- Flatten summary ---
        flat_summary = flatten_json(summary)

        row = {
            "session": session_dir.name,
            "rig": acq["instrument_id"],
            "mouse_n": acq["subject_id"],
            "experimenter": ",".join(acq.get("experimenters", [])),
            "day": day,
            "duration_min": round(duration_min, 1),
            **flat_summary
        }
        rows.append(row)
        if verbose: print(f"Loaded summary for {session_dir.name}")
        
    df = pd.DataFrame(rows)
    df = df.sort_values("day")
    df["session_idx"] = range(len(df))
    return df

# --- Plotting primitives ---
def plot_metric(df: pd.DataFrame, metric: str, x_col: str, tag_col: str, save_img_path: str | None = None) -> None:
    """Plot `metric` vs `x_col` for each mouse (grouped by 'mouse_n'), annotate with `tag_col` if present, and optionally save the figure."""
    fig, ax = plt.subplots(figsize = (14, 8))


    categories = df[x_col].drop_duplicates().tolist()
    pos_map = {c: i for i, c in enumerate(categories)} # Uniform spacing between x axis columns

    for inst, g in df.groupby("mouse_n"):
        g = g.sort_values(x_col)
        x = g[x_col].map(pos_map)  
        y = g[metric]
        line, = ax.plot(x, y, marker="o")
        color = line.get_color()
        if tag_col in g.columns: annotate_points(ax, x, y, g[tag_col], color=color)
    
    ax.set_title(f"{metric}")
    ax.set_ylabel(metric)
    ax.set_xlabel(x_col)
    ax.set_xticks(range(len(categories)))
    ax.set_xticklabels(categories, rotation=90)
    y_min, y_max = plt.ylim()
    plt.ylim(bottom = min(y_min, 0))
    plt.legend()
    plt.tight_layout()
    plt.show()
    if save_img_path: 
        fig.savefig(os.path.join(save_img_path, metric.replace('/', '-')), dpi=175, bbox_inches='tight')

def plot_series(df: pd.DataFrame, metric: str, cols: dict, x_col: str, tag_col: str, save_img_path: str | None = None) -> None:
    """Plot a per-mouse line of `cols['mean']` vs `x_col` with optional std errorbars and min/max band, annotate with `tag_col`, and optionally save."""
    fig, ax = plt.subplots(figsize = (14, 8)) 
    categories = df[x_col].drop_duplicates().tolist()
    pos_map = {c: i for i, c in enumerate(categories)}

    for inst, g in df.groupby("mouse_n"):
        g = g.sort_values(x_col)
        x = g[x_col].map(pos_map)  
        mean_vals = g[cols["mean"]]
        line, = ax.plot(x, mean_vals, marker="o", label=f"{inst}")
        color = line.get_color()
        if "std" in cols:
            ax.errorbar(x, mean_vals, yerr=g[cols["std"]], fmt="none", ecolor=color, capsize=3, alpha=0.8)
        if "min" in cols and "max" in cols:
            ax.fill_between(x, g[cols["min"]], g[cols["max"]], alpha=0.30, color = color)
        # annotate per mouse line
        if tag_col in g.columns: annotate_points(ax, x, mean_vals, g[tag_col], color=color)
        
    ax.set_xlabel(x_col)
    ax.set_ylabel(metric)
    categories = df[x_col].drop_duplicates().tolist()
    ax.set_xticks(range(len(categories)))
    ax.set_xticklabels(categories, rotation=90)

    y_min, y_max = plt.ylim()
    plt.ylim(bottom = min(y_min, 0))
    plt.legend()
    plt.tight_layout()
    plt.show()
    if save_img_path: 
        fig.savefig(os.path.join(save_img_path, f"a_{metric.replace('/', '-')}"), dpi=175, bbox_inches='tight')

def plot_slices(df: pd.DataFrame, metric: str, x_col: str, tag_col: str, save_img_path: str | None = None) -> None:
    """ Plot stacked slice bars of `{metric}` from columns named like `Slices_<start>-<end>_<metric>`    """

    prefix = "Slices_"
    full_col = f"Slices_Full_{metric}"

    # Collect slice columns for this metric (exclude 'Full')
    slice_cols = [c for c in df.columns if c.startswith(prefix) and c.endswith("_" + metric) and c != full_col]
    if not slice_cols:
        return

    # Helpers
    def _extract_label(col: str) -> str:
        return col[len(prefix):-len("_" + metric)]

    def _parse_bounds(label: str):
        a, b = label.split("-", 1)
        unit:str = "%" if (label.endswith("%")) else "min"
        a = a.replace("%", ""); b = b.replace("%", "")
        try:
            return float(a), float(b), unit
        except ValueError:
            return 0.0, 0.0, unit

    # Parse all slices (preserve column order)
    parsed = []  # (start, end, unit, label, column)
    for column in slice_cols:
        label = _extract_label(column)
        start, end, unit = _parse_bounds(label)
        parsed.append((start, end, unit, label, column))

    # Detect the slice width and the unit used in this DataFrame
    start, end, unit, _, _ = parsed[0]
    slice_width = end - start


    # Group columns by start bucket (rounded to int to absorb tiny float noise)
    groups = {}
    for s, _, _, _, c in parsed:
        key = int(round(s))
        groups.setdefault(key, []).append(c)

    # Prepare figure
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(df[x_col]))
    bottom = np.zeros(len(df), dtype=float)

    # Color map: one color per start bucket
    cmap = plt.get_cmap("tab20")
    def _color_for(i: int):
        return cmap(i % cmap.N)

    # Stack in ascending start order
    starts_sorted = sorted(groups.keys())
    for i, start_key in enumerate(starts_sorted):
        cols = groups[start_key]
        # Collapse multiple variants for same start (use max per-row to avoid double-counting)
        vals = df[cols].max(axis=1, skipna=True).values
        color = _color_for(i)
        start_lbl = int(start_key)
        end_lbl = int(round(start_key + slice_width))
        label = f"{start_lbl}-{end_lbl}{'%' if unit == '%' else ''}"
        ax.bar(x, vals, bottom=bottom, color=color, alpha=0.6, label=label)
        bottom[:] = bottom + np.nan_to_num(vals, nan=0.0)

    # Overlay one FULL line per mouse_n (no legend entry)
    if full_col in df.columns and "mouse_n" in df.columns:
        for mouse, g in df.groupby("mouse_n"):
            idx = df.index.get_indexer(g.index)
            line, = ax.plot(idx, g[full_col], marker="o", linestyle="-", linewidth=0)
            color = line.get_color()
            if tag_col in g.columns:
                annotate_points(ax, idx, g[full_col].values, g[tag_col], color=color)

    # Axes & legend
    ax.set_xticks(x)
    ax.set_xticklabels(df[x_col], rotation=90)
    ax.set_xlabel(x_col)
    ax.set_ylabel(f"{metric}")

    handles, labels = ax.get_legend_handles_labels()
    # De-duplicate labels (safety)
    seen, new_h, new_l = set(), [], []
    for h, l in zip(handles, labels):
        if l not in seen:
            seen.add(l); new_h.append(h); new_l.append(l)
    ax.legend(new_h, new_l, title=f"Slice ({unit})", ncol=1, frameon=True, loc="upper left", bbox_to_anchor=(1.02, 1.0))

    plt.tight_layout()
    plt.show()

    if save_img_path:
        fig.savefig(os.path.join(save_img_path, f"s_{metric.replace('/', '-')}"), dpi=175, bbox_inches='tight')

def scatter_by_rig_with_subcolumns(df: pd.DataFrame, metric:str, tag_col: str = None, save_img_path: str | None = None) -> None:
    """
    Scatter plot of `metric` organized by rig on x-axis, with each mouse_n as a sub-column within each rig.
    Plots all session points, plus the mean ± std for each (rig, mouse_n) subgroup.
    """
    # --- Prepare ordering: rigs by first appearance; mice by first appearance within each rig ---
    x_in_order = df["rig"].astype(str).drop_duplicates().tolist()

    # For consistent "subcolumns", collect mouse order per rig (appearance order within each rig)
    mice_per_rig = {}
    for rig in x_in_order:
        g = df[df["rig"].astype(str) == str(rig)]
        mice_per_rig[rig] = g["mouse_n"].astype(str).drop_duplicates().tolist()

    # --- Set up plot ---
    fig, ax = plt.subplots(figsize=(14, 8))
    color_list = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2',
         '#7f7f7f', '#bcbd22', '#17becf']

    # To hold tick positions and labels for the x-axis
    xtick_pos = []
    xtick_lab = []

    # --- Draw per rig ---
    for r_idx, rig in enumerate(x_in_order):
        rig_center = r_idx 
        mice = sorted(mice_per_rig[rig])
        n_mice = len(mice)

        # Precompute offsets for mice inside this rig, centered around rig_center
        offsets = []
        for i in range(n_mice):
            offsets.append((i - (n_mice - 1) / 2.0) /n_mice)

        # Draw a faint separator between rigs (except before the first one)
        if r_idx > 0:
            ax.axvline(rig_center - 0.5, color='lightgray', linestyle='--', linewidth=0.8, zorder=0)

        # --- For each mouse in this rig: scatter points + mean ± std ---
        for i, mouse in enumerate(mice):
            color = color_list[i%10]
            x_pos = rig_center + offsets[i]

            # Subgroup data
            sg = df[(df["rig"].astype(str) == str(rig)) & (df["mouse_n"].astype(str) == str(mouse))]
            y_vals = sg[metric].dropna()

            # Scatter all session points
            if not y_vals.empty:
                # Mean and std (errorbar)
                m = y_vals.mean()
                s = y_vals.std()  # NaN-safe; if only one point, std is NaN and errorbar won't draw
                ax.errorbar(x_pos, m, yerr=s, fmt='D', markersize=6, capsize=4, linewidth=1.2, zorder=3, color=color, alpha=0.65)

                # Scatter points (auto color cycle)
                ax.scatter([x_pos] * len(y_vals), y_vals.values, s=30, color=color)

            #  Annotations 
            if tag_col in g.columns: 
                dates = []
                for date in sg[tag_col]: 
                    dates.append(str(date)[5:]) 
                x = [x_pos] * len(y_vals)
                annotate_points(ax, x, y_vals, dates, color=color, fontsize=7)

            # Collect x tick for this subcolumn (two-line label: rig on top, mouse below)
            xtick_pos.append(x_pos)
            xtick_lab.append(f"{rig}\n{mouse}")
        
    # --- Axes cosmetics ---
    ax.set_xlabel("rig")
    ax.set_ylabel(metric)

    # x-ticks at each subcolumn center
    ax.set_xticks(xtick_pos)
    ax.set_xticklabels(xtick_lab, rotation=90)

    # y-baseline handling (optional: keep zero visible if relevant)
    y_min, y_max = ax.get_ylim()
    ax.set_ylim(bottom=min(0, y_min))

    fig.tight_layout()
    plt.show()

    if save_img_path:
        fig.savefig(os.path.join(save_img_path, f"sc_{metric.replace('/', '-')}"), dpi=175, bbox_inches='tight')

# --- Plotting Coordinator ---
def plot_grouped_metrics(df: pd.DataFrame, x_col: str = "day", tag_col: str = "rig", save_img_path: str | None = None) -> None:
    """
    Groups columns of a df depending on the type of metric (single, family or slice), then plots it accordingly. Optionally saves these figures.
        - Single metrics: Property that only has a value -> Basic line plot
        - Family Metrics: Columns that store the mean, std, min & max of the same property -> Plots it in a graph where the mean is the main line, 
                            the std is represented as err bars and the max and min are a shaded area.
        - Slice props: Properties that are stored in slices of the session -> These are drawn in a bar plot where the slices are on in top of each other
    """
    df = df.sort_values(by=x_col)

    stats_suffixes = ["mean", "std", "min", "max"]
    series_metrics, single_metrics, slice_properties = {}, [], set()

    for col in df.columns:
        for suffix in stats_suffixes:
            if col.endswith("_" + suffix):
                base = col[:-(len(suffix) + 1)]
                series_metrics.setdefault(base, {})[suffix] = col
                break
        else:
            if col.startswith("task"):
                single_metrics.append(col)
            elif col.startswith("Slices_"):
                slice_properties.add(col.split("_", 2)[-1])
    

    for base, cols in series_metrics.items():
        if "mean" in cols:
            scatter_by_rig_with_subcolumns(df, metric=cols["mean"], tag_col="day", save_img_path=save_img_path)
            plot_series(df, base, cols, x_col, tag_col, save_img_path=save_img_path)
    for metric in single_metrics:
        scatter_by_rig_with_subcolumns(df, metric, tag_col="day", save_img_path=save_img_path)
        plot_metric(df, metric, x_col, tag_col, save_img_path=save_img_path)
    for prop in slice_properties:
        scatter_by_rig_with_subcolumns(df, metric=f"Slices_Full_{prop}", tag_col="day", save_img_path=save_img_path)
        plot_slices(df, prop, x_col, tag_col, save_img_path=save_img_path)

    # scatter_by_rig_with_subcolumns(df, metric=metric)

df = load_json_summaries(tests_dir)    


plot_grouped_metrics(df, x_col = "day", tag_col = "rig", save_img_path=save_img_path)
#df.loc[:,["behavior_mean", "rig"]]